# 🧠 Evaluación Difusa de Riesgo Crediticio
### German Credit Dataset — UCI Machine Learning Repository

> **Objetivo:** Construir y comparar sistemas de inferencia difusa (FIS) para estimar el riesgo crediticio usando lógica difusa.

| Modelos | Variables de entrada | Reglas |
|---------|---------------------|--------|
| Mamdani, Takagi-Sugeno, Tsukamoto, Híbrido | Duración, Monto, Historial, % Cuota | 8 reglas justificadas |

---

## 📦 1. Instalación de Dependencias

In [ ]:
!pip install scikit-fuzzy scikit-learn pandas numpy matplotlib seaborn -q
print('✅ Dependencias instaladas')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
print('✅ Librerías importadas')

## 📥 2. Carga del Dataset

El German Credit Dataset (UCI id=144) tiene **1000 instancias** y **20 atributos**.  
Lo cargamos directo desde la URL oficial con nombres de columna explícitos.

In [ ]:
# ── Carga directa con nombres de columna explícitos ──────────────────────────
# La versión german.data usa códigos A11-A65; la cargamos asignando
# nombres semánticos directamente para evitar cualquier ambigüedad.

COL_NAMES = [
    'checking_status',    # A1  – Estado cuenta corriente
    'duration',           # A2  – Duración en meses  ★
    'credit_history',     # A3  – Historial crediticio  ★
    'purpose',            # A4  – Propósito del crédito
    'credit_amount',      # A5  – Monto del crédito  ★
    'savings_status',     # A6  – Estado de ahorros
    'employment',         # A7  – Empleo actual
    'installment_pct',    # A8  – % cuota vs ingreso  ★
    'personal_status',    # A9  – Estado personal / sexo
    'other_parties',      # A10 – Otros deudores / garantes
    'residence_since',    # A11 – Residencia actual (años)
    'property_magnitude', # A12 – Propiedad
    'age',                # A13 – Edad
    'other_payment_plans',# A14 – Otros planes de pago
    'housing',            # A15 – Tipo de vivienda
    'existing_credits',   # A16 – Créditos existentes
    'job',                # A17 – Tipo de empleo
    'num_dependents',     # A18 – Personas a cargo
    'own_telephone',      # A19 – Teléfono propio
    'foreign_worker',     # A20 – Trabajador extranjero
    'risk_label'          # Target: 1=bueno, 2=malo
]

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'

df = pd.read_csv(URL, sep=' ', header=None, names=COL_NAMES)

print(f'✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Verificación rápida ───────────────────────────────────────────────────────
print('Tipos de dato:')
print(df.dtypes)
print('\nValores nulos:', df.isnull().sum().sum())
print('\nDistribución del riesgo (1=bueno, 2=malo):')
print(df['risk_label'].value_counts())

## 🔧 3. Preprocesamiento

### Variables seleccionadas

| Variable | Columna | Tipo | Justificación |
|----------|---------|------|---------------|
| Duración del crédito | `duration` | Numérica (meses) | Proxy de exposición temporal al riesgo |
| Monto del préstamo | `credit_amount` | Numérica (DM) | Magnitud absoluta del riesgo financiero |
| Historial crediticio | `credit_history` | Categórica → ordinal | Variable con mayor AUC individual en literatura |
| % cuota del ingreso | `installment_pct` | Ordinal 1-4 | Mide capacidad de pago relativa al ingreso |

In [ ]:
# ── Mapeo del historial crediticio (A30-A34) a escala ordinal 0-4 ────────────
# A30: sin créditos / todos pagados          → 0 (peor)
# A31: todos los créditos pagados puntualmente → 1
# A32: créditos existentes al día            → 2
# A33: pagos diferidos en el pasado          → 3
# A34: cuenta crítica / otros créditos existentes → 4 (mejor para scoring)
HISTORY_MAP = {'A30': 0, 'A31': 1, 'A32': 2, 'A33': 3, 'A34': 4}

work_df = pd.DataFrame({
    'duration':       df['duration'].astype(float),
    'credit_amount':  df['credit_amount'].astype(float),
    'credit_history': df['credit_history'].map(HISTORY_MAP).astype(float),
    'installment_pct': df['installment_pct'].astype(float),
    # Target: 0 = bajo riesgo (bueno), 1 = alto riesgo (malo)
    'risk': (df['risk_label'] == 2).astype(int)
})

print('Dataset de trabajo — primeras filas:')
print(work_df.head())
print(f'\nValores nulos: {work_df.isnull().sum().sum()}')
print(f'\nEstadísticas descriptivas:')
work_df.describe().round(2)

In [ ]:
# ── Análisis exploratorio ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribución de Variables por Nivel de Riesgo', fontsize=15, fontweight='bold')

vars_plot = [
    ('duration',        'Duración del Crédito (meses)', axes[0,0]),
    ('credit_amount',   'Monto del Préstamo (DM)',      axes[0,1]),
    ('credit_history',  'Historial Crediticio (0-4)',   axes[1,0]),
    ('installment_pct', '% Ingreso para Cuotas (1-4)',  axes[1,1]),
]

for col, title, ax in vars_plot:
    ax.hist(work_df[work_df['risk']==0][col], bins=20, alpha=0.65,
            color='#2196F3', label='Bajo Riesgo', edgecolor='white')
    ax.hist(work_df[work_df['risk']==1][col], bins=20, alpha=0.65,
            color='#F44336', label='Alto Riesgo', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Frecuencia')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('exploratory_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Análisis exploratorio generado')

## 🔶 4. Funciones de Membresía

### Etiquetas lingüísticas

| Variable | Etiquetas | Rangos (aprox.) |
|----------|-----------|------------------|
| Duración | Corto / Medio / Largo | <24 / 12-48 / >36 meses |
| Monto | Bajo / Medio / Alto / Muy Alto | <3k / 2-9k / 7-15k / >13k DM |
| Historial | Malo / Regular / Bueno | 0-2 / 1-3 / 2-4 |
| % Cuota | Bajo / Moderado / Alto | cat 1 / cat 2-3 / cat 4 |
| **Riesgo** | **Bajo / Medio / Alto** | **0-45 / 30-70 / 55-100** |

In [ ]:
# ── Universos de discurso ─────────────────────────────────────────────────────
x_dur  = np.arange(4,   73,     1)      # meses
x_amt  = np.arange(250, 18500,  50)     # Deutsche Marks
x_his  = np.arange(0,   4.05,   0.05)   # ordinal 0-4
x_ins  = np.arange(1,   4.05,   0.05)   # categoría 1-4
x_risk = np.arange(0,   100.5,  0.5)    # score 0-100

# ── Duración ─────────────────────────────────────────────────────────────────
mf_dur_short  = fuzz.trimf(x_dur,  [4,  4,  24])
mf_dur_medium = fuzz.trimf(x_dur,  [12, 30, 48])
mf_dur_long   = fuzz.trapmf(x_dur, [36, 54, 72, 72])

# ── Monto ────────────────────────────────────────────────────────────────────
mf_amt_low      = fuzz.trapmf(x_amt, [250,  250,  1500, 3000])
mf_amt_medium   = fuzz.trimf(x_amt,  [2000, 5000, 9000])
mf_amt_high     = fuzz.trimf(x_amt,  [7000, 11000,15000])
mf_amt_veryhigh = fuzz.trapmf(x_amt, [13000,15000,18424,18424])

# ── Historial crediticio ──────────────────────────────────────────────────────
mf_his_bad  = fuzz.trapmf(x_his, [0, 0, 1, 2])
mf_his_fair = fuzz.trimf(x_his,  [1, 2, 3])
mf_his_good = fuzz.trapmf(x_his, [2, 3, 4, 4])

# ── % Cuota del ingreso ───────────────────────────────────────────────────────
mf_ins_low  = fuzz.trapmf(x_ins, [1, 1, 1.5, 2])
mf_ins_mod  = fuzz.trimf(x_ins,  [1.5, 2.5, 3.5])
mf_ins_high = fuzz.trapmf(x_ins, [3, 3.5, 4, 4])

# ── Riesgo (salida) ───────────────────────────────────────────────────────────
mf_risk_low  = fuzz.trapmf(x_risk, [0,  0,  25, 45])
mf_risk_med  = fuzz.trimf(x_risk,  [30, 50, 70])
mf_risk_high = fuzz.trapmf(x_risk, [55, 75, 100, 100])

print('✅ Funciones de membresía definidas')

In [ ]:
# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Funciones de Membresía — Sistema de Inferencia Difusa',
             fontsize=15, fontweight='bold', y=1.01)

def plot_mfs(ax, universe, mfs_labels, title, xlabel):
    colors = ['#1E88E5','#43A047','#FB8C00','#E53935']
    for (mf, label), c in zip(mfs_labels, colors):
        ax.plot(universe, mf, color=c, lw=2.5, label=label)
        ax.fill_between(universe, mf, alpha=0.10, color=c)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel(xlabel); ax.set_ylabel('μ (grado de membresía)')
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim(-0.05, 1.1)
    ax.spines[['top','right']].set_visible(False)

plot_mfs(axes[0,0], x_dur,
    [(mf_dur_short,'Corto'), (mf_dur_medium,'Medio'), (mf_dur_long,'Largo')],
    'Duración del Crédito', 'Meses')
axes[0,0].axvline(24, color='gray', ls='--', alpha=0.4)
axes[0,0].axvline(48, color='gray', ls='--', alpha=0.4)

plot_mfs(axes[0,1], x_amt,
    [(mf_amt_low,'Bajo'),(mf_amt_medium,'Medio'),(mf_amt_high,'Alto'),(mf_amt_veryhigh,'Muy alto')],
    'Monto del Préstamo', 'Deutsche Marks')

plot_mfs(axes[1,0], x_his,
    [(mf_his_bad,'Malo'),(mf_his_fair,'Regular'),(mf_his_good,'Bueno')],
    'Historial Crediticio', 'Score ordinal (0=crítico, 4=excelente)')
axes[1,0].set_xticks([0,1,2,3,4])
axes[1,0].set_xticklabels(['Crítico','Tardío','Al día','Buen hist.','Excelente'], fontsize=9)

plot_mfs(axes[1,1], x_ins,
    [(mf_ins_low,'Bajo <25%'),(mf_ins_mod,'Moderado'),(mf_ins_high,'Alto >75%')],
    '% Ingreso para Cuotas', 'Categoría (1-4)')
axes[1,1].set_xticks([1,2,3,4])
axes[1,1].set_xticklabels(['<25%','25-50%','50-75%','>75%'])

plot_mfs(axes[2,0], x_risk,
    [(mf_risk_low,'Bajo'),(mf_risk_med,'Medio'),(mf_risk_high,'Alto')],
    'Riesgo Crediticio (SALIDA)', 'Score de Riesgo (0–100)')
axes[2,0].axvline(33, color='gray', ls='--', alpha=0.4)
axes[2,0].axvline(66, color='gray', ls='--', alpha=0.4)

# Panel informativo
ax = axes[2,1]; ax.axis('off')
info = (
    'Resumen del Sistema Difuso\n'
    '─────────────────────────────────\n'
    'Variables de entrada: 4\n'
    '  • Duración     (3 etiquetas)\n'
    '  • Monto        (4 etiquetas)\n'
    '  • Historial    (3 etiquetas)\n'
    '  • % Cuota      (3 etiquetas)\n\n'
    'Variable de salida: 1\n'
    '  • Riesgo       (3 etiquetas)\n\n'
    'Reglas: 8\n'
    'Modelos: Mamdani | T-Sugeno\n'
    '         Tsukamoto | Híbrido\n'
    '─────────────────────────────────\n'
    'Dataset: German Credit (UCI id=144)\n'
    'Instancias: 1000  Atributos: 20'
)
ax.text(0.05, 0.95, info, transform=ax.transAxes, fontsize=11,
        va='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.9))

plt.tight_layout()
plt.savefig('membership_functions.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Visualización de funciones de membresía completada')

## 📏 5. Base de Reglas (8 reglas justificadas)

| # | Antecedente | Consecuente | Justificación |
|---|------------|-------------|---------------|
| R1 | Historial=**Malo** AND Duración=**Larga** | Riesgo=**Alto** | Peor combinación: sin historial positivo + máxima exposición temporal |
| R2 | Historial=**Bueno** AND Monto=**Bajo** | Riesgo=**Bajo** | Cliente confiable + exposición mínima → riesgo sistémico nulo |
| R3 | %Cuota=**Alta** AND Monto=**Alto** | Riesgo=**Alto** | Sobreendeudamiento relativo al ingreso disponible |
| R4 | Historial=**Bueno** AND %Cuota=**Baja** | Riesgo=**Bajo** | Perfil solvente con amplio margen de ingreso |
| R5 | Duración=**Larga** AND Monto=**Muy Alto** | Riesgo=**Alto** | Máxima exposición monetaria y temporal simultánea |
| R6 | Historial=**Regular** AND Duración=**Corta** | Riesgo=**Medio** | Historial neutro mitigado por plazo manejable |
| R7 | Historial=**Malo** AND Monto=**Bajo** AND Duración=**Corta** | Riesgo=**Medio** | Mal historial atenuado por baja exposición |
| R8 | Historial=**Bueno** AND Duración=**Corta** AND %Cuota=**Baja** | Riesgo=**Bajo** | Perfil ideal: solvencia + plazos cortos + bajo compromiso de ingreso |

## 🤖 6. Modelo Mamdani

In [ ]:
# ── Variables de control (skfuzzy) ───────────────────────────────────────────
v_dur = ctrl.Antecedent(x_dur,  'duration')
v_amt = ctrl.Antecedent(x_amt,  'amount')
v_his = ctrl.Antecedent(x_his,  'history')
v_ins = ctrl.Antecedent(x_ins,  'installment')
v_rsk = ctrl.Consequent(x_risk, 'risk')

v_dur['short']  = mf_dur_short;   v_dur['medium'] = mf_dur_medium; v_dur['long']   = mf_dur_long
v_amt['low']    = mf_amt_low;     v_amt['medium'] = mf_amt_medium; v_amt['high']   = mf_amt_high
v_amt['very_high'] = mf_amt_veryhigh
v_his['bad']  = mf_his_bad;  v_his['fair'] = mf_his_fair; v_his['good'] = mf_his_good
v_ins['low']  = mf_ins_low;  v_ins['moderate'] = mf_ins_mod; v_ins['high'] = mf_ins_high
v_rsk['low']  = mf_risk_low; v_rsk['medium'] = mf_risk_med; v_rsk['high'] = mf_risk_high
v_rsk.defuzzify_method = 'centroid'

# ── 8 Reglas ─────────────────────────────────────────────────────────────────
rules = [
    ctrl.Rule(v_his['bad']  & v_dur['long'],                         v_rsk['high']),   # R1
    ctrl.Rule(v_his['good'] & v_amt['low'],                          v_rsk['low']),    # R2
    ctrl.Rule(v_ins['high'] & v_amt['high'],                         v_rsk['high']),   # R3
    ctrl.Rule(v_his['good'] & v_ins['low'],                          v_rsk['low']),    # R4
    ctrl.Rule(v_dur['long'] & v_amt['very_high'],                    v_rsk['high']),   # R5
    ctrl.Rule(v_his['fair'] & v_dur['short'],                        v_rsk['medium']), # R6
    ctrl.Rule(v_his['bad']  & v_amt['low']   & v_dur['short'],       v_rsk['medium']), # R7
    ctrl.Rule(v_his['good'] & v_dur['short'] & v_ins['low'],         v_rsk['low']),    # R8
]

mamdani_cs  = ctrl.ControlSystem(rules)
mamdani_sim = ctrl.ControlSystemSimulation(mamdani_cs)
print('✅ Sistema Mamdani construido (8 reglas, defuzz=centroid)')

In [ ]:
# ── Evaluación sobre el dataset ───────────────────────────────────────────────
def infer_mamdani(dur, amt, his, ins):
    try:
        mamdani_sim.input['duration']    = float(np.clip(dur, 4, 72))
        mamdani_sim.input['amount']      = float(np.clip(amt, 250, 18424))
        mamdani_sim.input['history']     = float(np.clip(his, 0, 4))
        mamdani_sim.input['installment'] = float(np.clip(ins, 1, 4))
        mamdani_sim.compute()
        return float(mamdani_sim.output.get('risk', 50.0))
    except Exception:
        return 50.0

print('Evaluando Mamdani (1000 instancias)...')
work_df['mamdani_score'] = [
    infer_mamdani(r.duration, r.credit_amount, r.credit_history, r.installment_pct)
    for r in work_df.itertuples()
]
work_df['mamdani_pred'] = (work_df['mamdani_score'] > 50).astype(int)

acc_m = accuracy_score(work_df['risk'], work_df['mamdani_pred'])
print(f'✅ Mamdani — Accuracy: {acc_m:.4f} | Score μ={work_df["mamdani_score"].mean():.1f} σ={work_df["mamdani_score"].std():.1f}')

## 🤖 7. Modelo Takagi-Sugeno (orden 1)

In [ ]:
# ── Implementación manual T-S ─────────────────────────────────────────────────
# Consecuentes lineales: z_i = a0 + a1·dur_n + a2·amt_n + a3·his_n + a4·ins_n
# Defuzzificación: promedio ponderado  z* = Σ(w_i · z_i) / Σ(w_i)

def mu(val, universe, mf):
    return float(fuzz.interp_membership(universe, mf, float(val)))

def infer_ts(dur, amt, his, ins):
    dur = np.clip(float(dur), 4, 72)
    amt = np.clip(float(amt), 250, 18424)
    his = np.clip(float(his), 0, 4)
    ins = np.clip(float(ins), 1, 4)

    # Normalización [0,1] para los consecuentes lineales
    d = (dur - 4)   / (72   - 4)
    a = (amt - 250) / (18424-250)
    h = his / 4
    i = (ins-1) / 3

    # Grados de activación
    ds = mu(dur, x_dur, mf_dur_short);   dm = mu(dur, x_dur, mf_dur_medium);  dl = mu(dur, x_dur, mf_dur_long)
    al = mu(amt, x_amt, mf_amt_low);     am = mu(amt, x_amt, mf_amt_medium);  ah = mu(amt, x_amt, mf_amt_high);  avh= mu(amt, x_amt, mf_amt_veryhigh)
    hb = mu(his, x_his, mf_his_bad);     hf = mu(his, x_his, mf_his_fair);    hg = mu(his, x_his, mf_his_good)
    il = mu(ins, x_ins, mf_ins_low);     im = mu(ins, x_ins, mf_ins_mod);     ih = mu(ins, x_ins, mf_ins_high)

    # Reglas: (firing_strength, salida_lineal)
    rules_ts = [
        (min(hb, dl),      20 + 40*(1-h) + 30*d),        # R1  alto riesgo
        (min(hg, al),       5 + 10*a),                    # R2  bajo riesgo
        (min(ih, ah),      60 + 30*a + 10*i),             # R3  alto riesgo
        (min(hg, il),      10 + 5*a  + 5*d),              # R4  bajo riesgo
        (min(dl, avh),     70 + 25*d + 10*a),             # R5  alto riesgo
        (min(hf, ds),      40 + 15*a + 5*d),              # R6  medio riesgo
        (min(hb, al, ds),  35 + 10*a),                    # R7  medio riesgo
        (min(hg, ds, il),   8 + 7*a),                     # R8  bajo riesgo
    ]

    W = sum(w for w,_ in rules_ts)
    if W < 1e-10:
        return 50.0
    return float(np.clip(sum(w*z for w,z in rules_ts) / W, 0, 100))

print('Evaluando Takagi-Sugeno...')
work_df['ts_score'] = [
    infer_ts(r.duration, r.credit_amount, r.credit_history, r.installment_pct)
    for r in work_df.itertuples()
]
work_df['ts_pred'] = (work_df['ts_score'] > 50).astype(int)
acc_ts = accuracy_score(work_df['risk'], work_df['ts_pred'])
print(f'✅ Takagi-Sugeno — Accuracy: {acc_ts:.4f}')

## 🏆 8. BONUS: Tsukamoto y Modelo Híbrido

In [ ]:
# ── Tsukamoto FIS ─────────────────────────────────────────────────────────────
# Consecuentes: funciones de membresía MONOTÓNICAS (sigmoidales).
# Salida crisp obtenida invirtiendo la sigmoide.

def inv_sig_high(mu_val, a=0.12, c=70):   # monotónica creciente → riesgo alto
    mu_val = np.clip(float(mu_val), 0.001, 0.999)
    return float(np.clip(c + np.log(mu_val/(1-mu_val))/a, 0, 100))

def inv_sig_low(mu_val, a=0.12, c=30):    # monotónica decreciente → riesgo bajo
    mu_val = np.clip(float(mu_val), 0.001, 0.999)
    return float(np.clip(c - np.log(mu_val/(1-mu_val))/a, 0, 100))

def infer_tsukamoto(dur, amt, his, ins):
    dur = np.clip(float(dur), 4,   72)
    amt = np.clip(float(amt), 250, 18424)
    his = np.clip(float(his), 0,   4)
    ins = np.clip(float(ins), 1,   4)

    hb  = mu(his, x_his, mf_his_bad);   hg = mu(his, x_his, mf_his_good); hf = mu(his, x_his, mf_his_fair)
    dl  = mu(dur, x_dur, mf_dur_long);  ds = mu(dur, x_dur, mf_dur_short)
    al  = mu(amt, x_amt, mf_amt_low);   ah = mu(amt, x_amt, mf_amt_high); avh = mu(amt, x_amt, mf_amt_veryhigh)
    ih  = mu(ins, x_ins, mf_ins_high);  il = mu(ins, x_ins, mf_ins_low)

    # Reglas de alto riesgo  → sigmoide creciente
    high_rules = [min(hb, dl), min(ih, ah), min(dl, avh)]
    # Reglas de bajo riesgo  → sigmoide decreciente
    low_rules  = [min(hg, al), min(hg, il), min(hg, ds, il)]

    wz_pairs = (
        [(w, inv_sig_high(max(w, 0.001))) for w in high_rules] +
        [(w, inv_sig_low( max(w, 0.001))) for w in low_rules]
    )
    W = sum(w for w,_ in wz_pairs)
    if W < 1e-10:
        return 50.0
    return float(np.clip(sum(w*z for w,z in wz_pairs) / W, 0, 100))

print('Evaluando Tsukamoto...')
work_df['tsk_score'] = [
    infer_tsukamoto(r.duration, r.credit_amount, r.credit_history, r.installment_pct)
    for r in work_df.itertuples()
]
work_df['tsk_pred'] = (work_df['tsk_score'] > 50).astype(int)

# ── Híbrido (ensemble ponderado) ──────────────────────────────────────────────
work_df['hybrid_score'] = 0.40*work_df['mamdani_score'] + 0.40*work_df['ts_score'] + 0.20*work_df['tsk_score']
work_df['hybrid_pred']  = (work_df['hybrid_score'] > 50).astype(int)

acc_tsk = accuracy_score(work_df['risk'], work_df['tsk_pred'])
acc_hyb = accuracy_score(work_df['risk'], work_df['hybrid_pred'])
print(f'✅ Tsukamoto — Accuracy: {acc_tsk:.4f}')
print(f'✅ Híbrido   — Accuracy: {acc_hyb:.4f}')

## 📊 9. Evaluación y Tabla Comparativa

In [ ]:
# ── Métricas completas ────────────────────────────────────────────────────────
y_true = work_df['risk']

MODELS = {
    'Mamdani':       (work_df['mamdani_pred'], work_df['mamdani_score']),
    'Takagi-Sugeno': (work_df['ts_pred'],      work_df['ts_score']),
    'Tsukamoto':     (work_df['tsk_pred'],      work_df['tsk_score']),
    'Híbrido':       (work_df['hybrid_pred'],   work_df['hybrid_score']),
}

rows = []
for name, (preds, scores) in MODELS.items():
    scores_n = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    rows.append({
        'Modelo':    name,
        'Accuracy':  round(accuracy_score(y_true, preds), 4),
        'Precision': round(precision_score(y_true, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, preds, zero_division=0), 4),
        'F1-Score':  round(f1_score(y_true, preds, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_true, scores_n), 4),
    })

cmp_df = pd.DataFrame(rows).set_index('Modelo')

print('='*65)
print('          TABLA COMPARATIVA — MODELOS FIS')
print('='*65)
print(cmp_df.to_string())
print('='*65)
print(f"\n🏆 Mejor Accuracy : {cmp_df['Accuracy'].idxmax()}")
print(f"🏆 Mejor AUC-ROC  : {cmp_df['AUC-ROC'].idxmax()}")
print(f"🏆 Mejor F1-Score : {cmp_df['F1-Score'].idxmax()}")

In [ ]:
# ── Gráficas comparativas ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Comparativa de Modelos de Inferencia Difusa', fontsize=15, fontweight='bold')

METRICS  = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
MCOLORS  = ['#1E88E5','#43A047','#FB8C00','#8E24AA']
MNAMES   = cmp_df.index.tolist()

for idx, metric in enumerate(METRICS):
    ax = axes[idx//3][idx%3]
    vals = cmp_df[metric].values
    bars = ax.bar(MNAMES, vals, color=MCOLORS, width=0.55, edgecolor='white', lw=1.5)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1.1); ax.set_ylabel('Valor')
    ax.tick_params(axis='x', rotation=15)
    ax.axhline(0.7, color='gray', ls='--', alpha=0.5, lw=1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

# Gráfica 6: todas las métricas agrupadas
ax = axes[1][2]
x = np.arange(len(MNAMES))
w = 0.16
for i, (m, c) in enumerate(zip(METRICS[:4], ['#1E88E5','#43A047','#FB8C00','#E53935'])):
    ax.bar(x + i*w, cmp_df[m].values, w, label=m, color=c, alpha=0.85, edgecolor='white')
ax.set_title('Todas las métricas', fontweight='bold', fontsize=12)
ax.set_xticks(x + w*1.5); ax.set_xticklabels(MNAMES, rotation=15, fontsize=9)
ax.set_ylim(0, 1.1); ax.legend(fontsize=9)
ax.axhline(0.7, color='gray', ls='--', alpha=0.5)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Matrices de Confusión', fontsize=14, fontweight='bold')

for ax, (name, (preds, _)) in zip(axes, MODELS.items()):
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Bajo','Alto'], yticklabels=['Bajo','Alto'])
    ax.set_title(f'{name}\nAcc={accuracy_score(y_true,preds):.3f}', fontweight='bold')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Superficie de decisión T-S: Duración × Monto ─────────────────────────────
print('Generando superficie de decisión (puede tomar ~20 s)...')
D_range = np.linspace(4,  72,   30)
A_range = np.linspace(250,12000,30)
Z = np.zeros((len(D_range), len(A_range)))
for i, d in enumerate(D_range):
    for j, a in enumerate(A_range):
        Z[i, j] = infer_ts(d, a, 2, 2)  # historial=regular, cuota=moderada

fig, ax = plt.subplots(figsize=(11, 7))
DD, AA = np.meshgrid(A_range, D_range)
cf = ax.contourf(DD, AA, Z, levels=20, cmap='RdYlGn_r', alpha=0.85)
cl = ax.contour(DD, AA, Z, levels=[33,50,66], colors=['blue','black','red'], linewidths=2)
ax.clabel(cl, fmt='%d', fontsize=11)
plt.colorbar(cf, ax=ax, label='Score de Riesgo (0=bajo, 100=alto)')
ax.set_xlabel('Monto (DM)', fontsize=13); ax.set_ylabel('Duración (meses)', fontsize=13)
ax.set_title('Superficie de Decisión — Takagi-Sugeno\nHistorial=Regular, %Cuota=Moderado',
             fontsize=13, fontweight='bold')
ax.axhline(24, color='white', ls='--', alpha=0.7, lw=1.5)
ax.axhline(48, color='white', ls='-.', alpha=0.7, lw=1.5)
plt.tight_layout()
plt.savefig('decision_surface.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Superficie de decisión generada')

## 💬 10. Discusión Técnica y Conclusiones

In [ ]:
# ── Casos de prueba representativos ──────────────────────────────────────────
TEST = [
    ('Perfil A — Alto Riesgo',  60, 15000, 0, 4),
    ('Perfil B — Bajo Riesgo',  12,  1500, 4, 1),
    ('Perfil C — Riesgo Medio', 30,  5000, 2, 2),
    ('Perfil D — Borderline',   24,  8000, 1, 3),
]

print(f"{'Perfil':<28} {'Mamdani':>9} {'T-Sugeno':>9} {'Tsukamoto':>10} {'Híbrido':>9} {'Decisión':>12}")
print('-' * 82)
for nombre, d, a, h, i in TEST:
    sm = infer_mamdani(d, a, h, i)
    st = infer_ts(d, a, h, i)
    sk = infer_tsukamoto(d, a, h, i)
    sh = 0.4*sm + 0.4*st + 0.2*sk
    dec = '⚠️ ALTO' if sh > 50 else '✅ BAJO'
    print(f'{nombre:<28} {sm:>9.1f} {st:>9.1f} {sk:>10.1f} {sh:>9.1f} {dec:>12}')

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           DISCUSIÓN TÉCNICA — CONCLUSIONES                   ║
╠══════════════════════════════════════════════════════════════╣

1. MAMDANI
   ✔ Altamente interpretable: reglas con etiquetas lingüísticas
     comprensibles para analistas de crédito no técnicos.
   ✔ Defuzzificación por centroide produce salidas suaves.
   ✘ La agregación de consecuentes difusos es costosa y sensible
     al diseño de las MFs de salida.

2. TAKAGI-SUGENO (orden 1)
   ✔ Consecuentes lineales hacen el modelo más eficiente y
     compatible con optimización (ANFIS, gradient descent).
   ✔ Captura bien relaciones no lineales si los coeficientes
     están bien calibrados.
   ✘ Menos legible para expertos de negocio.

3. TSUKAMOTO
   ✔ Garantiza monotonía de la salida respecto al riesgo,
     propiedad deseable en scoring crediticio regulado.
   ✘ Restricción de monotonía limita la expresividad del modelo.
   ✘ Sensible al diseño de las funciones sigmoidales inversas.

4. MODELO HÍBRIDO (ensemble 40-40-20)
   ✔ Combina fortalezas de los tres modelos.
   ✔ Reduce varianza de predicción en casos fronterizos.
   ✘ Mayor opacidad: difícil justificar pesos ante reguladores.

LIMITACIONES GENERALES:
   • Las reglas y MFs se definieron heurísticamente;
     ANFIS automatizaría su ajuste mediante datos.
   • Dataset desbalanceado (70% bueno / 30% malo) favorece
     accuracy sobre recall en la clase minoritaria.
   • Umbral fijo en 50 es subóptimo; debería optimizarse
     según el costo asimétrico de falsos positivos/negativos.

╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
print('\n✅ Notebook completado exitosamente.')
print('   Archivos generados:')
import os
for f in ['exploratory_analysis.png','membership_functions.png',
          'model_comparison.png','confusion_matrices.png','decision_surface.png']:
    if os.path.exists(f):
        print(f'   ✔ {f}')